In [0]:
%sql
create catalog if not exists investment_pyspark;
use catalog investment_pyspark;

create schema if not exists bronze;
create volume if not exists bronze.landing;

In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
display(
    dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/holdings.csv/")
)

In [0]:
# 1. Source Path (Where your uploaded holdings.csv lives)
source_path = "/Volumes/investment_pyspark/bronze/landing/"

# 2. Schema Path (Databricks will create this '_schemas' folder automatically)
schema_path = "/Volumes/investment_pyspark/bronze/landing/_schemas/holdings"

# 3. Checkpoint Path (Databricks will create this '_checkpoints' folder automatically)
checkpoint_path = (
    "/Volumes/investment_pyspark/bronze/landing/_checkpoints/holdings"
)

In [0]:
df_raw = spark.readStream.format("cloudFiles").option(
    "cloudFiles.format", "csv"
).option(
    "cloudFiles.schemaLocation", schema_path
).option(
    "header", "true"
).option(
    "inferSchema", "true"
).load(source_path)

In [0]:
from pyspark.sql.functions import col
df_bronze = df_raw.withColumn("ingestion_timestamp",current_timestamp()).withColumn("source_file",col("_metadata.file_path"))

In [0]:
query = (
    df_bronze.writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)  # Prevents duplicate reads
    .outputMode("append")  # Appends new data
    .trigger(
        availableNow=True
    )  # Processes available files then shuts down
    .toTable("investment_pyspark.bronze.holdings_raw")
)

# Wait for the micro-batch to complete pro#cessing
query.awaitTermination()